In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [2]:
animes=pd.read_csv(r"C:\Users\gouab\Downloads\animes.csv")
ratings=pd.read_csv(r"C:\Users\gouab\Downloads\ratings.csv")

In [3]:
animes.head()


,animeID,title,alternative_title,type,year,score,episodes,mal_url,sequel,image_url,genres,genres_detailed
0,1,Howl's Moving Castle,Howl no Ugoku Shiro,MOVIE,2004,8.41,1,https://myanimelist.net/anime/431,False,https://cdn.myanimelist.net/images/anime/1470/...,"['Adventure', 'Award Winning', 'Drama', 'Fanta...","['action', 'adventure', 'age gap', 'air force'..."
1,2,Death Note,NaN,TV,2006,8.63,37,https://myanimelist.net/anime/1535,False,https://cdn.myanimelist.net/images/anime/1079/...,"['Supernatural', 'Suspense']","['achronological order', 'acting', 'adapted in..."
2,3,Problem Children Are Coming from Another World...,Mondaiji-tachi ga Isekai kara Kuru Sou desu yo?,TV,2013,7.42,10,https://myanimelist.net/anime/15315,False,https://cdn.myanimelist.net/images/anime/12/43...,"['Action', 'Comedy', 'Fantasy']","['action', 'alternative world', 'anthropomorph..."
3,4,BTOOOM!,Btooom!,TV,2012,7.34,12,https://myanimelist.net/anime/14345,False,https://cdn.myanimelist.net/images/anime/4/409...,"['Action', 'Sci-Fi', 'Suspense']","['achronological order', 'action', 'adventure'..."
4,5,Sword Art Online,NaN,TV,2012,7.5,25,https://myanimelist.net/anime/11757,False,https://cdn.myanimelist.net/images/anime/11/39...,"['Action', 'Adventure', 'Fantasy', 'Romance']","['action', 'action drama', 'adventure', 'alter..."


In [4]:
ratings.head(20)

,userID,animeID,rating
0,1,1,10
1,1,2,10
2,1,3,7
3,1,4,10
4,1,5,10
5,1,6,10
6,1,7,10
7,1,8,10
8,1,9,6
9,1,10,10


In [5]:
print(animes.shape)
print(ratings.shape)

(20237, 12)
(148170496, 3)


In [7]:
animes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20237 entries, 0 to 20236
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   animeID            20237 non-null  int64 
 1   title              20237 non-null  object
 2   alternative_title  8676 non-null   object
 3   type               20237 non-null  object
 4   year               20237 non-null  object
 5   score              20237 non-null  object
 6   episodes           20237 non-null  int64 
 7   mal_url            20237 non-null  object
 8   sequel             20237 non-null  bool  
 9   image_url          20237 non-null  object
 10  genres             20237 non-null  object
 11  genres_detailed    20237 non-null  object
dtypes: bool(1), int64(2), object(9)
memory usage: 1.7+ MB


In [8]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148170496 entries, 0 to 148170495
Data columns (total 3 columns):
 #   Column   Dtype
---  ------   -----
 0   userID   int64
 1   animeID  int64
 2   rating   int64
dtypes: int64(3)
memory usage: 3.3 GB


In [9]:
ratings['userID'].nunique()

1774522

In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix, vstack
import numpy as np

# Define Your Filters
MIN_ANIME_RATINGS = 50
MIN_USER_RATINGS = 10

# 1. Filter Animes
print("Filtering animes...")
anime_counts = ratings['animeID'].value_counts()
popular_animes = anime_counts[anime_counts >= MIN_ANIME_RATINGS].index

# 2. Filter Users
print("Filtering users...")
user_counts = ratings['userID'].value_counts()
active_users = user_counts[user_counts >= MIN_USER_RATINGS].index

# 3. Apply filters
filtered_ratings = ratings[
    (ratings['animeID'].isin(popular_animes)) & 
    (ratings['userID'].isin(active_users))
].copy()

print(f"Filtered ratings: {len(filtered_ratings)}")


BATCH_SIZE = 1000  # Process 1000 users at a time

# Get unique users and anime
unique_users = filtered_ratings['userID'].unique()
unique_anime = filtered_ratings['animeID'].unique()

print(f"Total users: {len(unique_users)}")
print(f"Total anime: {len(unique_anime)}")

# Create mapping for consistent indexing
user_to_idx = {user: idx for idx, user in enumerate(unique_users)}
anime_to_idx = {anime: idx for idx, anime in enumerate(unique_anime)}

# Map IDs to indices
filtered_ratings['user_idx'] = filtered_ratings['userID'].map(user_to_idx)
filtered_ratings['anime_idx'] = filtered_ratings['animeID'].map(anime_to_idx)

# Process in batches
sparse_matrices = []
num_batches = int(np.ceil(len(unique_users) / BATCH_SIZE))

for batch_num in range(num_batches):
    start_idx = batch_num * BATCH_SIZE
    end_idx = min((batch_num + 1) * BATCH_SIZE, len(unique_users))
    
    batch_users = unique_users[start_idx:end_idx]
    batch_data = filtered_ratings[filtered_ratings['userID'].isin(batch_users)]
    
    # Create sparse matrix for this batch
    row_indices = batch_data['user_idx'].values - start_idx # Adjust row indices for the batch
    col_indices = batch_data['anime_idx'].values
    data = batch_data['rating'].values
    
    batch_matrix = csr_matrix(
        (data, (row_indices, col_indices)),
        shape=(len(batch_users), len(unique_anime))
    )
    
    sparse_matrices.append(batch_matrix)
    print(f"Processed batch {batch_num + 1}/{num_batches}")

# Combine all batches
user_item_matrix_sparse = vstack(sparse_matrices)
print(f"\nFinal sparse matrix shape: {user_item_matrix_sparse.shape}")
print(f"Memory usage: {user_item_matrix_sparse.data.nbytes / (1024**2):.2f} MB")

Filtering animes...
Filtering users...
Filtered ratings: 145508395
Total users: 1366413
Total anime: 15172
Processed batch 1/1367
Processed batch 2/1367
Processed batch 3/1367
Processed batch 4/1367
Processed batch 5/1367
Processed batch 6/1367
Processed batch 7/1367
Processed batch 8/1367
Processed batch 9/1367
Processed batch 10/1367
Processed batch 11/1367
Processed batch 12/1367
Processed batch 13/1367
Processed batch 14/1367
Processed batch 15/1367
Processed batch 16/1367
Processed batch 17/1367
Processed batch 18/1367
Processed batch 19/1367
Processed batch 20/1367
Processed batch 21/1367
Processed batch 22/1367
Processed batch 23/1367
Processed batch 24/1367
Processed batch 25/1367
Processed batch 26/1367
Processed batch 27/1367
Processed batch 28/1367
Processed batch 29/1367
Processed batch 30/1367
Processed batch 31/1367
Processed batch 32/1367
Processed batch 33/1367
Processed batch 34/1367
Processed batch 35/1367
Processed batch 36/1367
Processed batch 37/1367
Processed batc

In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import matplotlib.pyplot as plt

# Load the anime details file
try:
    animes = pd.read_csv(r"C:\Users\gouab\Downloads\animes.csv")
    print(f"Loaded animes with columns: {animes.columns.tolist()}")
except FileNotFoundError:
    print("Error: 'animes.csv' not found. Please make sure it's in the same directory.")
    animes = pd.DataFrame(columns=['animeID', 'title', 'genres', 'score'])

# Create the new helper map for Title -> ID
name_to_id = pd.Series(animes.animeID.values, index=animes.title).to_dict()


sample_size = 1000000
if user_item_matrix_sparse.shape[0] > sample_size:
    indices = np.random.choice(user_item_matrix_sparse.shape[0], sample_size, replace=False)
    user_item_sample = user_item_matrix_sparse[indices]
else:
    user_item_sample = user_item_matrix_sparse

print(f"Testing on a sample of {user_item_sample.shape[0]} users...")

# Test a range of k values
possible_k_values = range(20, 1000, 10)
inertias = []
BATCH_SIZE = 1000

for k in possible_k_values:
    print(f"Testing k = {k}...")
    kmeans_test = MiniBatchKMeans(
        n_clusters=k, 
        random_state=42, 
        batch_size=BATCH_SIZE,
        n_init=3
    )
    kmeans_test.fit(user_item_sample)
    inertias.append(kmeans_test.inertia_)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(possible_k_values, inertias, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (WCSS)')
plt.title('The Elbow Method for Optimal k')
plt.grid(True)
plt.show()



In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
import pickle
name_to_id = pd.Series(animes.animeID.values, index=animes.title).to_dict()

N_CLUSTERS = 630 

print(f"\nCreating user clusters with N_CLUSTERS = {N_CLUSTERS}")
inertias = []
BATCH_SIZE = 1000
# Use MiniBatchKMeans instead of KMeans
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS, 
    random_state=42, 
    batch_size=BATCH_SIZE,
    max_iter=100,
    n_init=3
)

# Fit the model 
user_clusters = kmeans.fit_predict(user_item_matrix_sparse)

# Store cluster information
idx_to_user = {idx: user for user, idx in user_to_idx.items()}
user_cluster_mapping = {idx_to_user[idx]: cluster for idx, cluster in enumerate(user_clusters)}

print(f"Created {N_CLUSTERS} user clusters")
print(f"Cluster distribution: {np.bincount(user_clusters)}")

# Save the model 
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

print("Model saved")
def get_cluster_based_recommendations(watched_anime_ids, user_ratings_dict, n_recommendations=10):
    """
    Get recommendations for a new user based on their watched anime
    
    Parameters:
    - watched_anime_ids: List of anime IDs the user has watched
    - user_ratings_dict: Dictionary {anime_id: rating}
    - n_recommendations: Number of recommendations to return
    """
    from scipy.sparse import csr_matrix
    
    #Create SPARSE user profile vector 
    row_indices = []
    col_indices = []
    data = []
    
    for anime_id, rating in user_ratings_dict.items():
        if anime_id in anime_to_idx:
            anime_idx = anime_to_idx[anime_id]
            row_indices.append(0)  # Single row
            col_indices.append(anime_idx)
            data.append(rating)
    
    # Check if user has any valid animes
    if not data:
        print("User has no ratings for animes in our filtered database.")
        return pd.DataFrame(), None

    # Create sparse matrix 
    new_user_vector = csr_matrix(
        (data, (row_indices, col_indices)),
        shape=(1, len(unique_anime))
    )
    
    #Find the closest cluster
    print("\nStep 1: Finding most similar cluster...")
    cluster_id = kmeans.predict(new_user_vector)[0]
    print(f"Assigned to Cluster {cluster_id}")
    
    #Get users in that cluster
    users_in_cluster = [user_id for user_id, cluster in user_cluster_mapping.items() if cluster == cluster_id]
    print(f"Found {len(users_in_cluster)} users in this cluster")

    if not users_in_cluster:
        print("No other users found in this cluster.")
        return pd.DataFrame(), cluster_id
    
    #Calculate similarity with users in the cluster
    print("\nStep 2: Calculating similarity with cluster users...")
    cluster_user_indices = [user_to_idx[user_id] for user_id in users_in_cluster]
    cluster_user_matrix = user_item_matrix_sparse[cluster_user_indices]
    
    # Calculate cosine similarity
    similarities = cosine_similarity(new_user_vector, cluster_user_matrix).flatten()
    
    # Get top N similar users
    top_n_users = 50
    # Ensure we don't ask for more users than we have
    top_n_users = min(top_n_users, len(similarities))
    
    top_similar_indices = similarities.argsort()[::-1][:top_n_users]
    top_similar_scores = similarities[top_similar_indices]
    
    print(f"Top similarity score: {top_similar_scores[0]:.4f}")
    
    #Get weighted recommendations
    print("\nGenerating recommendations")
    similar_users_ratings = cluster_user_matrix[top_similar_indices]
    weighted_ratings = similar_users_ratings.T.dot(top_similar_scores)
    
    # Exclude already watched anime
    watched_indices = [anime_to_idx[anime_id] for anime_id in watched_anime_ids if anime_id in anime_to_idx]
    weighted_ratings[watched_indices] = -1
    
    # Get top recommendations
    top_anime_indices = weighted_ratings.argsort()[::-1][:n_recommendations]
    
    # Map to anime IDs and get details
    idx_to_anime = {idx: anime for anime, idx in anime_to_idx.items()}
    recommended_anime_ids = [idx_to_anime[idx] for idx in top_anime_indices]
    
    # Use the global 'animes' dataframe with correct column names
    recommendations = animes[animes['animeID'].isin(recommended_anime_ids)][['animeID', 'title', 'genres', 'score']].copy()
    
    # Add scores
    score_dict = {idx_to_anime[idx]: weighted_ratings[idx] for idx in top_anime_indices}
    recommendations['recommendation_score'] = recommendations['animeID'].map(score_dict)
    recommendations = recommendations.sort_values('recommendation_score', ascending=False)
    
    return recommendations, cluster_id



Creating user clusters with N_CLUSTERS = 630...
Created 630 user clusters
Cluster distribution: [   15     2    96  9343     1     4     1  6139     1  3980   946    13
 13979  8156  6555   451  1834  5829  3050  4418  9631  2383  6850  9097
     2   321    47  2094     1     3     1  7856  4379     6     3     1
    43     2  7012     2     1     1  1118   164     1     1    51     2
     2     1     1     1  1222     1     1  4808     1  9423     2     1
    88     1     1    41 15262 14188     1     9     5   601 18839  3933
     1     2 14035     1 11553    49     2  3259  3367  5226 10844   881
  4756     1  6342  9401  3325   101     2     1     1     1     1     3
     1     1     1  4532     1 18253   489  8324 15130   164  4825     1
     3  4335     2     2     2    10  6087     2   269     1  1238    49
     2     1     1     1     2     2     1     2 12748   778     1     1
  2183     2     1  7619     1  1196  1969  7513 11305  4683     3     1
     1   208    44     1   

In [ ]:
new_user_watched_by_name = {
    'Cowboy Bebop': 9,
    'Naruto': 8,
    'Fullmetal Alchemist: Brotherhood': 10,
    'Death Note': 7,
}

#Convert names to IDs
new_user_watched_by_id = {}
for name, rating in new_user_watched_by_name.items():
    if name in name_to_id:
        new_user_watched_by_id[name_to_id[name]] = rating
    else:
        print(f"Warning: Anime '{name}' not found in database. Skipping.")

# This is what the function needs
watched_ids = list(new_user_watched_by_id.keys())

print(f"\n{'='*80}")
print(f"NEW USER PROFILE:")
print(f"{'='*80}")
for name, rating in new_user_watched_by_name.items():
    if name in name_to_id:
        print(f"- {name}: Rating {rating}")

print(f"\n{'='*80}")
print(f"GENERATING CLUSTER-BASED RECOMMENDATIONS:")
print(f"{'='*80}")

recommendations, assigned_cluster = get_cluster_based_recommendations(
    watched_ids, 
    new_user_watched_by_id, 
    n_recommendations=10
)

print(f"\n{'='*80}")
print(f"TOP 10 RECOMMENDATIONS:")
print(f"{'='*80}")
print(recommendations)

In [ ]:
# Save all necessary data for Streamlit app
import pickle
with open('user_item_matrix.pkl', 'wb') as f:
    pickle.dump({
        'matrix': user_item_matrix_sparse,
        'user_to_idx': user_to_idx,
        'anime_to_idx': anime_to_idx,
        'unique_anime': unique_anime,
        'user_cluster_mapping': user_cluster_mapping
    }, f)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Create TF-IDF matrix for anime content
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(animes['genres'].fillna(''))
# Compute cosine similarity matrix
content_similarity_matrix = cosine_similarity(tfidf_matrix, dense_output=False)

In [ ]:
import pickle

with open('content_similarity.pkl', 'wb') as f:
    pickle.dump({
        'similarity_matrix': content_similarity_matrix,
        'tfidf_matrix': tfidf_matrix
    }, f)

print("Content similarity matrix saved!")